# MazimaCheck: is this face forged? (OpenForensics, face crops)

**Before you run anything: File > Save a copy in Drive.** Work in your copy, not this one.
Then Runtime > Change runtime type > **T4 GPU**.

**What this notebook does.** Our first model asks "is the whole picture AI-generated?". This one asks a different question about *each face*: "is this face real, or swapped or generated?". Think of it as a second inspector who only looks at faces, with a magnifying glass.

1. Download the validation part of OpenForensics (about 3 GB) and read its labels.
2. Cut out each face the same way the app does.
3. Check for shortcuts (face size, file size) before training, the same lesson as the first project.
4. Train ResNet-18 and ResNet-50 and compare them.
5. Test on faces the model never saw, and again after chat-app style damage.
6. Export an ONNX file the web app can run.

The dataset is used for academic purposes only, which fits coursework. Cite: Le, Nguyen, Yamagishi, Echizen, ICCV 2021.

In [ ]:
import os, io, json, random, zipfile, subprocess, time, math
import numpy as np, pandas as pd
import torch, torch.nn as nn
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression

DRY = os.environ.get("MZ_DRY") == "1"          # DRY is only for testing the code on fake data
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev, torch.cuda.get_device_name(0) if dev == "cuda" else "(no GPU: Runtime > Change runtime type > T4 GPU)")

ARCHS = ["resnet18"] if DRY else ["resnet18", "resnet50"]
EPOCHS = 1 if DRY else 6
BATCH = 16 if DRY else 64
SIZE, PAD, MINSIDE = 224, 0.20, 32               # PAD and SIZE must match the app (faceCrop in app.js)
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive"); OUT = "/content/drive/MyDrive/mazimacheck_faces"; WORK = "/content/of"
except Exception:
    OUT = os.environ.get("MZ_OUT", "./mazimacheck_faces"); WORK = os.environ.get("MZ_WORK", "./of")
os.makedirs(OUT, exist_ok=True); os.makedirs(WORK, exist_ok=True)
print("results are saved in:", OUT)

## 1. Get the data
This downloads straight from Zenodo into Colab's temporary disk (not your Drive), so it takes a few minutes and costs you no storage. If it stops, run the cell again: it skips what it already has.

In [ ]:
import requests
BASE = "https://zenodo.org/records/5528418/files/"
def fetch(name):
    # shows a progress bar, and if it stops you can simply run this cell again: it carries on from where it got to
    dst, url = f"{WORK}/{name}", BASE + name + "?download=1"
    total = int(requests.head(url, allow_redirects=True).headers.get("Content-Length", 0))
    have = os.path.getsize(dst) if os.path.exists(dst) else 0
    if total and have == total: print(name, "already complete"); return
    if total and have > total: os.remove(dst); have = 0
    with requests.get(url, stream=True, headers={"Range": f"bytes={have}-"} if have else {}) as r:
        if r.status_code == 416: print(name, "already complete"); return        # we already have all of it
        r.raise_for_status()
        if have and r.status_code != 206: have = 0
        with open(dst, "ab" if have else "wb") as f, tqdm(total=total or None, initial=have, unit="B", unit_scale=True, desc=name) as bar:
            for ch in r.iter_content(1 << 20): f.write(ch); bar.update(len(ch))
    if total: assert os.path.getsize(dst) == total, "incomplete download: run this cell again"
if not DRY:
    fetch("Val_poly.json"); fetch("Val.zip")
    if not os.path.isdir(f"{WORK}/Val"):
        assert zipfile.is_zipfile(f"{WORK}/Val.zip"), "Val.zip is incomplete: run this cell again"
        print("unzipping (a minute or two)...")
        zipfile.ZipFile(f"{WORK}/Val.zip").extractall(f"{WORK}/Val")
print(os.listdir(WORK))

## 2. Read the labels (and do not trust our guesses)
The annotation file is in COCO format. We print its categories and map them to real (0) or forged (1) by their **names**, and stop if the names are not what we expect.

In [ ]:
coco = json.load(open(f"{WORK}/Val_poly.json"))
print("keys:", list(coco.keys()))
print("categories:", coco["categories"])
print("images:", len(coco["images"]), "annotations:", len(coco["annotations"]))
print("first image:", coco["images"][0])
print("first annotation:", {k: (v if k != "segmentation" else "...") for k, v in coco["annotations"][0].items()})

LABEL = {}
for c in coco["categories"]:
    n = c["name"].lower()
    LABEL[c["id"]] = 1 if ("fake" in n or "forg" in n) else (0 if "real" in n else None)
print("label map (0 real, 1 forged):", {c["name"]: LABEL[c["id"]] for c in coco["categories"]})
assert set(LABEL.values()) == {0, 1}, "Category names are not what we expected. Read the printout above and fix LABEL by hand."

## 3. Cut out the faces
Same recipe as the app: take the face box, add 20% of its width and height on every side, then resize to 224 x 224. Faces whose shorter side is under 32 pixels are skipped because there is nothing to judge. Crops are saved as PNG so no extra compression sneaks in.

In [ ]:
path_of = {}
for r, _, fs in os.walk(f"{WORK}/Val"):
    for f in fs: path_of[f] = os.path.join(r, f)
imgs = {i["id"]: i for i in coco["images"]}
by_img = {}
for a in coco["annotations"]:
    if LABEL.get(a["category_id"]) is None or "bbox" not in a: continue
    by_img.setdefault(a["image_id"], []).append(a)

CROPS = f"{WORK}/crops"; os.makedirs(CROPS, exist_ok=True)
rows, skipped = [], 0
for iid, anns in tqdm(by_img.items()):
    meta = imgs[iid]; p = path_of.get(os.path.basename(meta["file_name"]))
    if p is None: skipped += len(anns); continue
    im = Image.open(p).convert("RGB"); W, H = im.size; src_bytes = os.path.getsize(p)
    for k, a in enumerate(anns):
        x, y, w, h = a["bbox"]
        if min(w, h) < MINSIDE: skipped += 1; continue
        x0, y0 = max(0, x - PAD * w), max(0, y - PAD * h)
        x1, y1 = min(W, x + w + PAD * w), min(H, y + h + PAD * h)
        cp = f"{CROPS}/{iid}_{k}.png"
        im.crop((int(x0), int(y0), int(x1), int(y1))).resize((SIZE, SIZE), Image.BILINEAR).save(cp)
        rows.append(dict(path=cp, image_id=iid, label=LABEL[a["category_id"]], w=w, h=h, srcW=W, srcH=H, src_bpp=src_bytes / (W * H)))
df = pd.DataFrame(rows)
print(f"{len(df)} face crops, {skipped} skipped (too small or image missing)")
print(df.label.value_counts().rename({0: "real", 1: "forged"}))

## 4. Shortcut audit (before training)
Last time, a model could partly guess "AI or real" from the picture's shape and file size without looking at content. Here we ask the same question for faces: **how well can face size and file size alone separate real from forged?** If that score (AUC) is far above 0.5, a model could cheat, so we balance the training set by face size.

We also split by **image**, never by face, so two faces from one photo never land on both sides of the exam.

In [ ]:
ids = sorted(df.image_id.unique()); rng = random.Random(SEED); rng.shuffle(ids)
n = len(ids); tr_ids, va_ids = set(ids[: int(.70 * n)]), set(ids[int(.70 * n): int(.85 * n)])
df["split"] = df.image_id.map(lambda i: "train" if i in tr_ids else ("val" if i in va_ids else "test"))
print(df.groupby(["split", "label"]).size().unstack())

df["logarea"] = np.log(df.w * df.h); df["aspect"] = df.w / df.h
def feats(d, cols): return d[cols].values
audits = {"face size (area, aspect)": ["logarea", "aspect"], "photo size + file bytes/pixel": ["srcW", "srcH", "src_bpp"],
          "all of these": ["logarea", "aspect", "srcW", "srcH", "src_bpp"]}
for name, cols in audits.items():
    tr, te = df[df.split == "train"], df[df.split == "test"]
    mu, sd = feats(tr, cols).mean(0), feats(tr, cols).std(0) + 1e-9
    clf = LogisticRegression(max_iter=1000).fit((feats(tr, cols) - mu) / sd, tr.label)
    print(f"{name:34s} AUC on test = {roc_auc_score(te.label, clf.predict_proba((feats(te, cols) - mu) / sd)[:, 1]):.3f}   (0.5 means no shortcut)")

In [ ]:
def balance_by_size(d, bins=10, seed=SEED):
    # inside each face-size bin, keep equal numbers of real and forged, so size says nothing about the label
    d = d.copy(); d["bin"] = pd.qcut(d.logarea, bins, labels=False, duplicates="drop"); out = []
    for _, g in d.groupby("bin"):
        k = int(g.label.value_counts().reindex([0, 1], fill_value=0).min())
        if k == 0: continue                                   # a bin with only one class cannot be balanced
        out += [g[g.label == c].sample(k, random_state=seed) for c in (0, 1)]
    return pd.concat(out).drop(columns="bin")
train_df = balance_by_size(df[df.split == "train"])
val_df, test_df = df[df.split == "val"], df[df.split == "test"]
test_bal = balance_by_size(test_df)
print("train after balancing:", train_df.label.value_counts().to_dict(), "| test:", len(test_df), "| size-balanced test:", len(test_bal))
tr_cols = ["logarea", "aspect"]; mu, sd = feats(train_df, tr_cols).mean(0), feats(train_df, tr_cols).std(0) + 1e-9
clf = LogisticRegression(max_iter=1000).fit((feats(train_df, tr_cols) - mu) / sd, train_df.label)
print("face-size AUC on the balanced test set:", round(roc_auc_score(test_bal.label, clf.predict_proba((feats(test_bal, tr_cols) - mu) / sd)[:, 1]), 3))
pc = ["srcW", "srcH", "src_bpp"]; mu2, sd2 = feats(train_df, pc).mean(0), feats(train_df, pc).std(0) + 1e-9
clf2 = LogisticRegression(max_iter=1000).fit((feats(train_df, pc) - mu2) / sd2, train_df.label)
print("photo size + bytes/pixel AUC after balancing (test):", round(roc_auc_score(test_bal.label, clf2.predict_proba((feats(test_bal, pc) - mu2) / sd2)[:, 1]), 3),
      "<- if this is still far above 0.5, say so in the report: face-size balancing does not remove it")

Here is a quick look at what the model will see. Top row real, bottom row forged. Do the forged ones look obviously fake to *you*? Write that down for the report.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 8, figsize=(16, 4.4))
for r, lab in enumerate((0, 1)):
    s = train_df[train_df.label == lab].sample(min(8, (train_df.label == lab).sum()), random_state=1)
    for c, p in enumerate(s.path):
        ax[r, c].imshow(Image.open(p)); ax[r, c].axis("off")
    ax[r, 0].set_title("real" if lab == 0 else "forged", loc="left")
plt.tight_layout(); plt.show()

## 5. Train
We train ResNet-18 and ResNet-50 the same way, so the comparison is fair. Training pictures are randomly flipped, colour-shifted, shrunk and re-compressed, so the model has seen chat-app damage before the exam. The best epoch is chosen on the **validation** faces, never on the test faces.

In [ ]:
import torchvision as tv
from torch.utils.data import Dataset, DataLoader

def damage(im, shrink=None, quality=None):
    if shrink: im = im.resize((shrink, shrink), Image.BILINEAR).resize((SIZE, SIZE), Image.BILINEAR)
    if quality:
        b = io.BytesIO(); im.save(b, "JPEG", quality=quality); b.seek(0); im = Image.open(b).convert("RGB")
    return im
norm = tv.transforms.Compose([tv.transforms.ToTensor(), tv.transforms.Normalize(MEAN, STD)])
jitter = tv.transforms.ColorJitter(.2, .2, .2)

class Faces(Dataset):
    def __init__(self, d, train=False, fixed=None): self.p, self.y, self.train, self.fixed = list(d.path), list(d.label), train, fixed
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        im = Image.open(self.p[i]).convert("RGB")
        if self.train:
            if random.random() < .5: im = im.transpose(Image.FLIP_LEFT_RIGHT)
            im = jitter(im)
            im = damage(im, random.randint(96, SIZE) if random.random() < .5 else None, random.randint(40, 95) if random.random() < .5 else None)
        elif self.fixed: im = damage(im, *self.fixed)
        return norm(im), torch.tensor(float(self.y[i]))
def loader(d, train=False, fixed=None):
    return DataLoader(Faces(d, train, fixed), batch_size=BATCH, shuffle=train, num_workers=0 if DRY else 2, drop_last=False)

def make(arch):
    m = getattr(tv.models, arch)(weights=None if DRY else "DEFAULT"); m.fc = nn.Linear(m.fc.in_features, 1); return m.to(dev)

@torch.no_grad()
def predict(m, dl):
    m.eval(); P = []
    for x, _ in dl: P.append(torch.sigmoid(m(x.to(dev)).squeeze(1)).float().cpu())
    return torch.cat(P).numpy()
def best_threshold(p, y):
    ts = np.linspace(.05, .95, 91); f = [f1_score(y, p >= t, average="macro") for t in ts]; return float(ts[int(np.argmax(f))]), float(max(f))

results, models = {}, {}
for arch in ARCHS:
    m = make(arch); opt = torch.optim.AdamW(m.parameters(), lr=2e-4, weight_decay=1e-4)
    dl = loader(train_df, train=True); sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-4, total_steps=EPOCHS * len(dl))
    scaler = torch.amp.GradScaler(enabled=(dev == "cuda")); lossf = nn.BCEWithLogitsLoss(); best = (-1, None, None)
    t0 = time.time()
    for ep in range(EPOCHS):
        m.train(); tot = 0
        for x, y in tqdm(dl, desc=f"{arch} epoch {ep + 1}/{EPOCHS}", leave=False):
            x, y = x.to(dev), y.to(dev)
            with torch.autocast(device_type=dev, enabled=(dev == "cuda")): loss = lossf(m(x).squeeze(1), y)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); tot += loss.item() * len(x)
        pv = predict(m, loader(val_df)); th, f1 = best_threshold(pv, val_df.label.values)
        print(f"{arch} epoch {ep + 1}: train loss {tot / len(train_df):.3f} | val macro-F1 {f1:.3f} at threshold {th:.2f} | val AUC {roc_auc_score(val_df.label, pv):.3f} | {time.time() - t0:.0f}s")
        if f1 > best[0]: best = (f1, th, {k: v.detach().cpu().clone() for k, v in m.state_dict().items()})
    m.load_state_dict(best[2]); models[arch] = m; results[arch] = dict(val_f1=best[0], threshold=best[1])
    torch.save(best[2], f"{WORK}/{arch}.pt")

## 6. The exam: faces the model has never seen
Same test faces for both models. We also re-test after damage (a small, heavily compressed copy, like a face from a forwarded WhatsApp video) and on the **size-balanced** test set, where face size cannot help.

In [ ]:
def report(m, d, th, fixed=None):
    p = predict(m, loader(d, fixed=fixed)); y = d.label.values
    return dict(f1=f1_score(y, p >= th, average="macro"), auc=roc_auc_score(y, p), cm=confusion_matrix(y, p >= th), p=p)
table = []
for arch, m in models.items():
    th = results[arch]["threshold"]
    for name, d, fx in [("test, as cut", test_df, None), ("test, size-balanced", test_bal, None),
                        ("test, damaged (112 px, JPEG q45)", test_df, (112, 45)), ("test, JPEG q75 only", test_df, (None, 75))]:
        r = report(m, d, th, fx); results[arch][name] = {k: v for k, v in r.items() if k != "p"}
        table.append(dict(model=arch, test=name, macroF1=round(r["f1"], 3), AUC=round(r["auc"], 3)))
        if name == "test, as cut": results[arch]["test_p"] = r["p"]
print(pd.DataFrame(table).to_string(index=False))
for arch in models:
    print(f"\n{arch} confusion matrix on the test faces (rows real/forged, columns predicted real/forged), threshold {results[arch]['threshold']:.2f}")
    print(results[arch]["test, as cut"]["cm"])

## 7. Pick a model, set the verdict zones, export for the app
We pick the model with the better **validation** score. If ResNet-18 is close to ResNet-50, prefer ResNet-18: it is about half the download. The zones work like the photo model: a score below `real_below` reads "likely real", above `ai_from` reads "likely forged", and the middle is "uncertain". They are chosen on validation faces so that each side is right about 80% of the time, when that is possible.

In [ ]:
pick = max(results, key=lambda a: results[a]["val_f1"]); m = models[pick].eval().cpu(); th = results[pick]["threshold"]
print("chosen:", pick, results[pick]["val_f1"])
pv = predict(models[pick].to(dev), loader(val_df)); yv = val_df.label.values
def zones(p, y, target=.80):
    hi = [t for t in np.linspace(.5, .99, 99) if (p >= t).sum() >= .05 * len(p) and y[p >= t].mean() >= target]
    lo = [t for t in np.linspace(.01, .5, 99) if (p < t).sum() >= .05 * len(p) and (1 - y[p < t]).mean() >= target]
    return (max(lo) if lo else .45), (min(hi) if hi else .65)
real_below, forged_from = zones(pv, yv); print(f"zones: likely real below {real_below:.2f}, likely forged from {forged_from:.2f}")

class Wrap(nn.Module):
    def __init__(s, m): super().__init__(); s.m = m
    def forward(s, x): return s.m(x).squeeze(-1)
onnx_path = f"{OUT}/faces_{pick}.onnx"; dummy = torch.randn(2, 3, SIZE, SIZE); w = Wrap(m).eval()
kw = dict(input_names=["input"], output_names=["logit"], dynamic_axes={"input": {0: "n"}, "logit": {0: "n"}}, opset_version=17)
try: torch.onnx.export(w, dummy, onnx_path, dynamo=False, **kw)
except TypeError: torch.onnx.export(w, dummy, onnx_path, **kw)
subprocess.run(["pip", "install", "-q", "onnxruntime"], check=False)
import onnxruntime as ort
s = ort.InferenceSession(onnx_path); x = torch.randn(4, 3, SIZE, SIZE)
diff = float(np.abs(s.run(None, {"input": x.numpy()})[0] - w(x).detach().numpy()).max())
print(f"ONNX saved: {onnx_path}  ({os.path.getsize(onnx_path) / 1e6:.1f} MB), max logit difference vs PyTorch {diff:.2e}")

summary = dict(model=pick, onnx=os.path.basename(onnx_path), crop=dict(size=SIZE, pad=PAD, min_side=MINSIDE), threshold=th,
               zones=dict(real_below=real_below, forged_from=forged_from), onnx_parity_max_logit_diff=diff,
               compared={a: {k: (v if not isinstance(v, dict) else {kk: (vv.tolist() if hasattr(vv, "tolist") else vv) for kk, vv in v.items()}) for k, v in r.items() if k != "test_p"} for a, r in results.items()},
               data=dict(source="OpenForensics v1.0.0 Val split", faces=int(len(df)), train=int(len(train_df)), val=int(len(val_df)), test=int(len(test_df))))
json.dump(summary, open(f"{OUT}/faces_summary.json", "w"), indent=1, default=float)
df.drop(columns=["path"]).to_csv(f"{OUT}/faces_split.csv", index=False)
print("saved faces_summary.json and faces_split.csv in", OUT)

## 8. Optional: a second dataset the model has never met
The honest test is a different source. Put faces in your Drive like this and run the cell:

`MyDrive/mazimacheck_faces/external/real/*.jpg` and `MyDrive/mazimacheck_faces/external/fake/*.jpg`

Faces should already be cropped. Anything you find must be allowed for research use. Report the result whatever it is: a drop is a finding, not a failure.

In [ ]:
import glob
EXT = f"{OUT}/external"
files = [(p, lab) for lab, nm in ((0, "real"), (1, "fake")) for p in glob.glob(f"{EXT}/{nm}/*")]
if not files: print("No external folder yet. Skipping.")
else:
    rows = []
    for p, lab in files:
        try: im = Image.open(p).convert("RGB").resize((SIZE, SIZE), Image.BILINEAR); q = f"{WORK}/ext_{len(rows)}.png"; im.save(q); rows.append(dict(path=q, label=lab))
        except Exception as e: print("skip", p, e)
    ext = pd.DataFrame(rows); r = report(models[pick].to(dev), ext, th)
    print(f"external faces: {len(ext)}  macro-F1 {r['f1']:.3f}  AUC {r['auc']:.3f}"); print(r["cm"])